# OpenIRM — AIRS Autoencoder Training (Google Colab Free-Tier GPU Fallback)

This notebook provides a self-contained, GPU-accelerated training pipeline for the **AIRS (Adaptive Insider Risk System)** PyTorch Autoencoder. Use this notebook on Google Colab if local hardware resources are limited.

In [ ]:
# Step 1: Install required dependencies
!pip install -q torch pandas scikit-learn fastparquet pyyaml joblib matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import matplotlib.pyplot as plt

# Step 2: GPU Device Detection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on training device: {device}")

In [ ]:
# Step 3: Define Symmetric AIRS PyTorch Autoencoder (72 -> 48 -> 24 -> 12 -> 24 -> 48 -> 72)
class AIRSAutoencoder(nn.Module):
    def __init__(self, input_dim=72, hidden_dims=[48, 24], latent_dim=12, dropout_rate=0.1):
        super().__init__()
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.LeakyReLU(0.1),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = h_dim
        encoder_layers.extend([nn.Linear(prev_dim, latent_dim), nn.LeakyReLU(0.1)])
        self.encoder = nn.Sequential(*encoder_layers)

        # Decoder
        decoder_layers = []
        prev_dim = latent_dim
        for h_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.LeakyReLU(0.1),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = h_dim
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        return self.decoder(self.encoder(x))

print("AIRS Autoencoder Architecture defined.")

In [ ]:
# Step 4: Synthetic / Uploaded Data Loader (Simulating Benign Training Split)
np.random.seed(42)
num_benign_samples = 5000
input_dim = 72

# Generate synthetic benign baseline feature matrix (scaled z-scores around 0.0)
x_benign = np.random.normal(loc=0.0, scale=1.0, size=(num_benign_samples, input_dim)).astype(np.float32)

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_benign)

# DataLoader split: 80% train, 20% validation
split_idx = int(0.8 * num_benign_samples)
t_train = torch.tensor(x_scaled[:split_idx], dtype=torch.float32)
t_val = torch.tensor(x_scaled[split_idx:], dtype=torch.float32)

train_loader = DataLoader(TensorDataset(t_train), batch_size=128, shuffle=True)
val_loader = DataLoader(TensorDataset(t_val), batch_size=128, shuffle=False)
print(f"Prepared {len(train_loader.dataset)} training samples and {len(val_loader.dataset)} validation samples.")

In [ ]:
# Step 5: Training Loop
model = AIRSAutoencoder().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.MSELoss()
epochs = 30

train_losses, val_losses = [], []

for epoch in range(1, epochs + 1):
    model.train()
    t_sum = 0.0
    for (batch_x,) in train_loader:
        batch_x = batch_x.to(device)
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_x)
        loss.backward()
        optimizer.step()
        t_sum += loss.item() * len(batch_x)
    train_loss = t_sum / len(train_loader.dataset)
    train_losses.append(train_loss)

    model.eval()
    v_sum = 0.0
    with torch.no_grad():
        for (batch_x,) in val_loader:
            batch_x = batch_x.to(device)
            out = model(batch_x)
            v_sum += criterion(out, batch_x).item() * len(batch_x)
    val_loss = v_sum / len(val_loader.dataset)
    val_losses.append(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{epochs:02d} | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}")

In [ ]:
# Step 6: Plot Training Loss History
plt.figure(figsize=(8, 4))
plt.plot(range(1, epochs + 1), train_losses, label="Train MSE Loss")
plt.plot(range(1, epochs + 1), val_losses, label="Val MSE Loss")
plt.xlabel("Epoch")
plt.ylabel("Reconstruction Loss (MSE)")
plt.title("AIRS Autoencoder Training Curve (Colab GPU)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Step 7: Export Checkpoints
torch.save({'model_state_dict': model.state_dict(), 'final_val_loss': val_losses[-1]}, 'airs_autoencoder.pt')
joblib.dump(scaler, 'airs_scaler.pkl')
print("Checkpoints 'airs_autoencoder.pt' and 'airs_scaler.pkl' successfully exported!")